# Crypto Transaction Monitoring — EDA & Pipeline Validation

**Project:** Fraud AI Investigator — MENA Fintech Portfolio  
**Notebook:** `notebooks/crypto_monitoring.ipynb`  
**Author:** Ahmed Raza  
**Last updated:** 2026-05

---

## Objective

Validate and analyse the crypto monitoring module. This notebook answers:

1. **Is the mixer detection logic working correctly?** — validate all 3 detection layers
2. **What does a sanctioned address look like on-chain?** — live Etherscan data for known mixers
3. **How does the risk scoring distribute across addresses?** — score distribution chart
4. **How does crypto monitoring complement the payment pipeline?** — integration analysis

## Why crypto monitoring matters for UAE/MENA fintech

The UAE is a global crypto hub — Dubai has one of the highest crypto adoption rates
globally and the Virtual Assets Regulatory Authority (VARA) was established in 2022.
CBUAE and VARA both require VASPs (Virtual Asset Service Providers) to:
- Screen wallet addresses against OFAC sanctions lists
- Detect mixer/tumbler interactions (Travel Rule compliance)
- Report suspicious on-chain activity alongside traditional STRs

This module adds exactly this capability to the existing payment AML pipeline.

## Detection layers

```
Wallet address
     │
     ├── Layer 1: Known mixer address blacklist  (OFAC SDN — instant)
     ├── Layer 2: Behavioural pattern analysis   (Etherscan data)
     │     ├── Round denomination amounts        (Tornado Cash pools)
     │     ├── Rapid in/out pattern              (layering)
     │     └── High internal tx ratio            (contract hopping)
     └── Layer 3: Composite risk score 0-100
           ├── >= 60  → CryptoAlert created → PENDING
           ├── >= 70  → LLM triage (Phase 3 pipeline)
           └── >= 90  → AWAITING_HUMAN immediately
```

## Prerequisites

```bash
# Add to .env:
# ETHERSCAN_API_KEY=your_key_here   ← from etherscan.io/myapikey (free)

# Start API:
uv run uvicorn app.main:app --reload

# Install notebook deps:
uv pip install matplotlib pandas requests

# Launch notebook:
uv run jupyter notebook notebooks/crypto_monitoring.ipynb
```

## Limitations

- Etherscan free tier: 5 req/sec — large address batches take time
- Mixer list is a point-in-time snapshot — new addresses require updating
- Behavioural patterns may produce false positives for high-frequency traders
- On-chain data only covers Ethereum — Polygon/Base/BSC need separate calls

---
## Section 0 — Environment setup

In [ ]:
import sys
import json
import warnings
from datetime import datetime
from pathlib import Path

import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings('ignore', category=DeprecationWarning)

plt.style.use('seaborn-v0_8-whitegrid')

BASE_URL        = 'http://localhost:8000'
SCREENSHOTS_DIR = PROJECT_ROOT / 'doc' / 'Screenshots'
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Run timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

---
## Section 1 — API health check

In [ ]:
try:
    health = requests.get(f'{BASE_URL}/health', timeout=5).json()
except Exception:
    raise RuntimeError('API not running. Start: uv run uvicorn app.main:app --reload')

assert health['status'] == 'ok'

print('API HEALTH')
print('=' * 45)
print(f"  Status      : {health['status']}")
print(f"  Gemini      : {'✓' if health['llm_providers']['gemini']     else '✗ missing'}")
print(f"  Groq        : {'✓' if health['llm_providers']['groq']       else '✗ missing'}")
print(f"  Etherscan   : {'✓' if health['llm_providers']['etherscan']  else '✗ missing'}")

etherscan_available = health['llm_providers'].get('etherscan', False)
if not etherscan_available:
    print()
    print('  Etherscan key not configured.')
    print('  Get free key: https://etherscan.io/myapikey')
    print('  Add to .env: ETHERSCAN_API_KEY=your_key')
    print('  Section 4+ requires live API key. Sections 2-3 work offline.')

---
## Section 2 — Known sanctioned mixer addresses

**Purpose:** Inspect the Layer 1 detection database.
These are OFAC-sanctioned mixer addresses — any interaction with them scores 70+ immediately.

**Regulatory context:** OFAC sanctioned Tornado Cash on August 8, 2022 — the first time
a smart contract (not a person or company) was added to the SDN list. This was a landmark
moment for crypto AML compliance. UAE VARA and CBUAE both require VASPs to screen against
these addresses as part of Travel Rule compliance.

In [ ]:
mixers = requests.get(f'{BASE_URL}/v1/crypto/mixers').json()

print(f'SANCTIONED MIXER DATABASE')
print('=' * 65)
print(f'Total entries: {mixers["count"]}')
print()

for m in mixers['mixers']:
    print(f"  {m['name']}")
    print(f"    Address : {m['address']}")
    print(f"    Sanction: {m['sanction']}  |  Listed: {m['date_listed']}")
    print(f"    Notes   : {m['notes']}")
    print()

---
## Section 3 — Offline detection validation

**Purpose:** Validate all 3 detection layers using synthetic transaction data.
No Etherscan API key needed here — we build fake transactions that represent
known mixer patterns and confirm the detector scores them correctly.

In [ ]:
from app.crypto.mixer_detector import MixerDetector, SANCTIONED_MIXER_ADDRESSES

detector = MixerDetector(score_threshold=60)

TORNADO_CASH = '0xd90e2f925da726b50c4ed8d0fb90ad053324f31b'
TEST_ADDR    = '0xd8da6bf26964af9d7eed9e03e53415d37aa96045'

# Test case 1: Direct mixer interaction
tx_to_mixer = {
    'hash': '0xtest1', 'from': TEST_ADDR, 'to': TORNADO_CASH,
    'value': str(int(1.0 * 1e18)), 'timeStamp': '1700000000', 'isError': '0',
}

# Test case 2: Clean transaction
tx_clean = {
    'hash': '0xtest2', 'from': TEST_ADDR, 'to': '0x1234567890123456789012345678901234567890',
    'value': str(int(0.137 * 1e18)), 'timeStamp': '1700003600', 'isError': '0',
}

# Test case 3: Rapid in/out (layering)
tx_in  = {'hash': '0xin',  'from': '0x9999999999999999999999999999999999999999', 'to': TEST_ADDR,
          'value': str(int(5.0 * 1e18)), 'timeStamp': '1700000000', 'isError': '0'}
tx_out = {'hash': '0xout', 'from': TEST_ADDR, 'to': '0x8888888888888888888888888888888888888888',
          'value': str(int(4.9 * 1e18)), 'timeStamp': '1700001800', 'isError': '0'}  # 30min later

test_cases = [
    ('Direct mixer interaction (TC Router)',    [tx_to_mixer],    True),
    ('Clean normal transaction',               [tx_clean],       False),
    ('Rapid in/out layering pattern',          [tx_in, tx_out],  True),
    ('Empty wallet — no transactions',         [],               False),
]

print('DETECTION VALIDATION')
print('=' * 65)
all_correct = True
for desc, txs, expected_flagged in test_cases:
    result  = detector.analyse(address=TEST_ADDR, transactions=txs,
                               token_transactions=[], eth_balance=1.0)
    correct = result.is_flagged == expected_flagged
    if not correct:
        all_correct = False
    status = '✓' if correct else '✗ FAIL'
    flag   = '🔴 FLAGGED' if result.is_flagged else '🟢 CLEAR'
    print(f'  {status} {desc}')
    print(f'     {flag} | score={result.risk_score} | severity={result.severity}')
    if result.signals:
        for s in result.signals:
            print(f'     → {s.signal_type}: {s.description[:70]}')
    print()

print(f'All detection cases: {"PASSED ✓" if all_correct else "FAILED ✗"}')

---
## Section 4 — Live Etherscan screening (requires API key)

**Purpose:** Screen known addresses using real Etherscan data.
We use two addresses:
- Tornado Cash Router (OFAC sanctioned) — should score high
- vitalik.eth — should score low (public figure, no mixer interaction)

**Skip this section** if Etherscan key is not configured — all previous sections still work.

In [ ]:
if not etherscan_available:
    print('Skipping live Etherscan screening — no API key configured')
    print('Add ETHERSCAN_API_KEY to .env and restart to run this section')
else:
    test_addresses = [
        {
            'address'    : '0xd90e2f925da726b50c4ed8d0fb90ad053324f31b',
            'label'      : 'Tornado Cash Router (OFAC sanctioned)',
            'expect_high': True,
        },
        {
            'address'    : '0xd8da6bf26964af9d7eed9e03e53415d37aa96045',
            'label'      : 'vitalik.eth (should be clean)',
            'expect_high': False,
        },
    ]

    live_results = []
    for test in test_addresses:
        print(f'Screening: {test["label"]}')
        resp = requests.post(
            f'{BASE_URL}/v1/crypto/screen',
            json={'address': test['address'], 'customer_id': 'NOTEBOOK-TEST', 'limit': 50},
            timeout=30,
        )
        if resp.status_code == 200:
            data   = resp.json()
            screen = data.get('screening', {})
            score  = screen.get('risk_score', 0)
            flag   = screen.get('is_flagged', False)
            print(f'  Score: {score} | Flagged: {flag} | Severity: {screen.get("severity")}')
            live_results.append({
                'label': test['label'], 'score': score,
                'flagged': flag, 'expected_high': test['expect_high'],
            })
        else:
            print(f'  Error: {resp.status_code} — {resp.json().get("detail")}')
        print()

    print(f'Live screening complete — {len(live_results)} addresses screened')

---
## Section 5 — Risk score distribution visualisation

In [ ]:
# Simulate a range of scores to show the scoring model visually
# In production this would use real screening results from app/data/crypto/screening_results.json

import numpy as np

crypto_path = PROJECT_ROOT / 'app' / 'data' / 'crypto' / 'screening_results.json'

if crypto_path.exists():
    with open(crypto_path) as f:
        screening_data = json.load(f)
    scores = [
        r.get('screening', {}).get('risk_score', 0)
        for r in screening_data.get('results', [])
        if r.get('screening')
    ]
    title_suffix = f'(live data — {len(scores)} addresses)'
else:
    # Synthetic score distribution for visualisation
    np.random.seed(42)
    scores = list(np.concatenate([
        np.random.normal(10,  8,  70).clip(0, 35),   # clean addresses
        np.random.normal(55, 12,  20).clip(35, 75),  # medium risk
        np.random.normal(80, 10,  10).clip(60, 100), # high risk / mixers
    ])).copy()
    scores = [int(min(max(s, 0), 100)) for s in scores]
    title_suffix = '(synthetic distribution for illustration)'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram with band zones
ax = axes[0]
ax.hist(scores, bins=20, color='#1565C0', alpha=0.75, edgecolor='white', zorder=3)
ax.axvspan(0,  60, alpha=0.06, color='#4CAF50', label='CLEAR (<60)')
ax.axvspan(60, 70, alpha=0.06, color='#FF9800', label='MEDIUM (60-70)')
ax.axvspan(70, 90, alpha=0.06, color='#F44336', label='HIGH (70-90)')
ax.axvspan(90, 100,alpha=0.06, color='#880E4F', label='CRITICAL (90+)')
ax.axvline(x=60, color='#FF9800', linestyle='--', linewidth=1.5, label='Alert threshold (60)')
ax.set_title('Crypto risk score distribution', fontsize=12)
ax.set_xlabel('Mixer detection score (0-100)')
ax.set_ylabel('Address count')
ax.legend(fontsize=8)

# Right: severity band breakdown
ax2 = axes[1]
bands = {
    'CLEAR\n(<60)': sum(1 for s in scores if s < 60),
    'MEDIUM\n(60-70)': sum(1 for s in scores if 60 <= s < 70),
    'HIGH\n(70-90)': sum(1 for s in scores if 70 <= s < 90),
    'CRITICAL\n(90+)': sum(1 for s in scores if s >= 90),
}
colors = ['#4CAF50', '#FF9800', '#F44336', '#880E4F']
bars   = ax2.bar(bands.keys(), bands.values(), color=colors, alpha=0.85, edgecolor='white')
for bar, (band, count) in zip(bars, bands.items()):
    pct = count / len(scores) * 100 if scores else 0
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{count}\n({pct:.0f}%)', ha='center', fontsize=10)
ax2.set_title('Address count by severity band', fontsize=12)
ax2.set_ylabel('Count')

plt.suptitle(
    f'Crypto Mixer Detection — Score Distribution {title_suffix}',
    y=1.02, fontsize=12, fontweight='bold'
)
plt.tight_layout()
save_path = SCREENSHOTS_DIR / '08_crypto_risk_scores.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved: {save_path}')

---
## Section 6 — Integration with payment pipeline

**Purpose:** Show how crypto alerts flow through the same pipeline as payment alerts.
Crypto alerts use `AlertTrigger.SANCTIONED_CORRIDOR` — the same trigger as payment
sanctions screening — so they flow through triage, HITL, and audit automatically.
No pipeline changes needed in Phases 4-7.

In [ ]:
# Show all alerts in store — payment and crypto mixed together
all_alerts = requests.get(f'{BASE_URL}/v1/alerts?limit=50').json()

print('PIPELINE INTEGRATION CHECK')
print('=' * 50)
print(f'Total alerts in store: {all_alerts["total"]}')
print()

if all_alerts['total'] > 0:
    for a in all_alerts['alerts'][:5]:
        source = 'CRYPTO' if a['tx_id'].startswith('CRYPTO') else 'PAYMENT'
        print(f"  [{source}] {a['trigger']} | {a['status']} | tx={a['tx_id'][:20]}")
    if all_alerts['total'] > 5:
        print(f'  ... and {all_alerts["total"] - 5} more')
else:
    print('  No alerts in store yet.')
    print('  Run: POST /v1/alerts/generate  (payment alerts)')
    print('  Run: POST /v1/crypto/screen    (crypto alerts)')

print()
print('Both payment and crypto alerts use the same:')
print('  ✓ AlertStore  — unified storage')
print('  ✓ AuditEvent  — full evidence trail')
print('  ✓ /v1/triage  — LLM scoring (Phase 3)')
print('  ✓ HITL queue  — analyst review (Phase 5)')

---
## Section 7 — Completion checklist

In [ ]:
print('CRYPTO MONITORING NOTEBOOK — COMPLETION CHECKLIST')
print('=' * 55)

checks = {
    'API health check passed'              : health['status'] == 'ok',
    'Mixer database endpoint works'        : requests.get(f'{BASE_URL}/v1/crypto/mixers').status_code == 200,
    'Status endpoint works'                : requests.get(f'{BASE_URL}/v1/crypto/status').status_code == 200,
    'Detection validation passed'          : all_correct,
    'Risk score chart saved'               : (SCREENSHOTS_DIR / '08_crypto_risk_scores.png').exists(),
    'Tornado Cash in mixer database'       : any('Tornado' in m['name'] for m in mixers['mixers']),
    'Etherscan API configured'             : etherscan_available,
}

all_passed = True
for label, passed in checks.items():
    print(f"  {'✓' if passed else '✗'}  {label}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed — crypto monitoring notebook complete ✓')
    print('Ready for Phase 4: LangGraph CryptoAgent integration.')
else:
    print('Some checks incomplete — see sections above.')
    if not etherscan_available:
        print('\n  To enable live Etherscan screening:')
        print('  1. https://etherscan.io/myapikey (free account)')
        print('  2. Add ETHERSCAN_API_KEY=your_key to .env')
        print('  3. Restart the API')

print(f'\nCompleted: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')